# 17. Scaled Ontology-Checked Text-to-SPARQL on Spider4SSC: One-Shot vs Error Retry vs Ontology Checker

**Presentation map:** Slide 27 (an ontology defines classes, properties, domains and ranges), Slides 43–44 (ground
entities first; the agent queries the graph), Slide 50 (validation), Slides 59–61 (deterministic ontology checks, LLM
repair, "unknown" after three failures, read-only gate).

Notebook 7 showed the checker on **15 questions** over one helpdesk graph (accuracy 20% → 67%). This notebook re-tests
it on **Spider4SSC** (Mendeley Data `4dnvv92trn`, 2025): the Spider text-to-SQL benchmark translated into **SPARQL**
over RDF versions of the same databases, with gold SPARQL whose results were verified to equal the SQL results.

- **Questions:** the full dev split — 608 questions over 20 databases (concerts, pets, cars, flights, world
  countries, …) — minus the 36 questions of `wta_1`, whose 45 MB graph is too slow for the in-memory engine used here
  → **572 questions over 19 graphs**.
- **Ontology:** each graph ships an RDF schema (`*.rdf-schema.json`): classes (tables), datatype properties with their
  range (`string`, `integer`, …) and object properties (foreign keys) with domain and range.

**Three systems, same model and the same full schema in the prompt:**

| System | What happens after the first query |
|---|---|
| (a) **One-shot** | the first query is executed as is |
| (b) **Error / empty retry** (strong baseline) | if the query does not parse, fails to execute or returns no rows, that is sent back; ≤ 3 retries |
| (c) **Ontology checker** | a deterministic checker (no LLM) tests syntax, the read-only gate, vocabulary, domain/range/direction, literal datatypes and grounding of string constants; violations are explained back; ≤ 3 repairs, then **"unknown"** |

**Measure:** execution accuracy — the predicted query returns the same multiset of rows as the gold query (values
compared after normalisation, column order ignored). For (c) we also count "unknown" answers: a system that says
"unknown" is not right, but it is not confidently wrong either.

**Model:** `gpt-6-luna`, no reasoning, via the OpenAI Batch API (`smash_batch.py`); each repair round is one Batch
job. Cached in `data/benchmarks/spider4ssc/cache/`.

In [1]:
import json, os, re, difflib, multiprocessing as mp
from collections import Counter, defaultdict
import numpy as np
import rdflib
from rdflib import Literal, URIRef, Variable, XSD
from rdflib.plugins.sparql import prepareQuery
from rdflib.plugins.sparql.parser import parseUpdate
from smash_batch import LLM, mode_from_env, wilson, paired_boot

MODE = mode_from_env()
TEST = MODE == "test"
BASE = "data/benchmarks/spider4ssc"
llm = LLM(f"{BASE}/cache" + ("_test" if TEST else ""))

dev = [dict(q, qid=f"q{i:03d}") for i, q in enumerate(json.load(open(f"{BASE}/dev.json"))) if q["db_id"] != "wta_1"]
if TEST:
    dev = [q for db in sorted({q["db_id"] for q in dev})[:4] for q in [x for x in dev if x["db_id"] == db][:4]]
DBS = sorted({q["db_id"] for q in dev})
GRAPHS, SCHEMA = {}, {}
for db in DBS:
    g = rdflib.Graph(); g.parse(f"{BASE}/database/{db}/{db}.ttl")
    GRAPHS[db] = g
    SCHEMA[db] = json.load(open(f"{BASE}/database/{db}/{db}.rdf-schema.json"))
print(f"{len(dev)} questions over {len(DBS)} graphs; triples per graph: "
      + ", ".join(f"{db} {len(GRAPHS[db]):,}" for db in DBS[:6]) + ", …")

572 questions over 19 graphs; triples per graph: battle_death 210, car_1 6,498, concert_singer 212, course_teach 111, cre_Doc_Template_Mgt 340, dog_kennels 699, …


## The ontology of each graph, as the model sees it

In [2]:
def prefixes(db):
    p = SCHEMA[db]["Prefixes"]
    return "".join(f"PREFIX {k}: <{v}>\n" for k, v in p.items()) + "PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>\n"

def qname(db, uri):
    best = max(((k, v) for k, v in SCHEMA[db]["Prefixes"].items() if uri.startswith(v)), key=lambda kv: len(kv[1]), default=None)
    return f"{best[0]}:{uri[len(best[1]):]}" if best else f"<{uri}>"

def is_obj(rng):
    return any(r.startswith("http") for r in rng)

def schema_text(db):
    S = SCHEMA[db]
    lines = ["Classes: " + ", ".join(qname(db, c) for c in S["Classes"])]
    for p, v in sorted(S["Properties"].items()):
        dom = "/".join(qname(db, d) for d in v["domain"])
        rng = "/".join(qname(db, r) if r.startswith("http") else f"xsd:{r}" for r in v["range"])
        lines.append(f"- {qname(db, p)}: {dom} -> {rng}" + ("  (link to another resource)" if is_obj(v["range"]) else ""))
    return "\n".join(lines)

print(f"Ontology of {DBS[0]}:\n{schema_text(DBS[0])}")

Ontology of battle_death:
Classes: :ship, :battle, :death
- battle:bulgarian_commander: :battle -> xsd:string
- battle:date: :battle -> xsd:string
- battle:id: :battle -> xsd:integer
- battle:latin_commander: :battle -> xsd:string
- battle:name: :battle -> xsd:string
- battle:result: :battle -> xsd:string
- death:CAUSED_BY_SHIP_ID: :death -> :ship  (link to another resource)
- death:caused_by_ship_id: :death -> xsd:integer
- death:id: :death -> xsd:integer
- death:injured: :death -> xsd:integer
- death:killed: :death -> xsd:integer
- death:note: :death -> xsd:string
- ship:LOST_IN_BATTLE: :ship -> :battle  (link to another resource)
- ship:disposition_of_ship: :ship -> xsd:string
- ship:id: :ship -> xsd:integer
- ship:location: :ship -> xsd:string
- ship:lost_in_battle: :ship -> xsd:integer
- ship:name: :ship -> xsd:string
- ship:ship_type: :ship -> xsd:string
- ship:tonnage: :ship -> xsd:string


## Executing queries safely

Every query runs in a forked process with a 20-second limit (the in-memory engine cannot be interrupted otherwise).
Results are normalised: numbers rounded to 4 decimals, IRIs reduced to their last segment, each row turned into a
sorted tuple so that column order does not matter.

In [3]:
CTX = mp.get_context("fork")
def _worker(db, q, conn):
    try:
        rows = []
        for r in GRAPHS[db].query(q):
            rows.append(tuple(sorted(norm_val(v) for v in r)))
        conn.send(("ok", rows))
    except Exception as e:
        conn.send(("error", f"{type(e).__name__}: {str(e)[:200]}"))

def norm_val(v):
    if v is None:
        return ""
    if isinstance(v, Literal):
        py = v.toPython()
        if isinstance(py, bool):
            return str(py).lower()
        if isinstance(py, (int, float)) or (isinstance(py, str) and re.fullmatch(r"-?\d+(\.\d+)?", py.strip() or "x")):
            try:
                return f"{round(float(py), 4):g}"
            except Exception:
                pass
        return str(py).strip()
    return str(v).rstrip("/").split("/")[-1]

def execute(db, q, timeout=20):
    a, b = CTX.Pipe(False)
    p = CTX.Process(target=_worker, args=(db, prefixes(db) + q, b)); p.start()
    if a.poll(timeout):
        status, out = a.recv(); p.join()
        return status, out
    p.kill(); p.join()
    return "error", f"Timeout: the query ran longer than {timeout} s"

GOLD = {}
for q in dev:
    GOLD[q["qid"]] = execute(q["db_id"], q["sparql"])
bad = [q["qid"] for q in dev if GOLD[q["qid"]][0] != "ok"]
dev = [q for q in dev if q["qid"] not in bad]
print(f"Gold queries executed: {len(GOLD) - len(bad)} ok, {len(bad)} failed (dropped) -> {len(dev)} questions")
same = lambda a, b: Counter(a) == Counter(b)

Gold queries executed: 563 ok, 9 failed (dropped) -> 563 questions


## The deterministic ontology checker (no LLM)

1. **Syntax / read-only gate** — the query must parse as a SPARQL `SELECT`/`ASK`; updates are refused.
2. **Vocabulary** — every class and property must exist in the ontology (close names are suggested).
3. **Domain / range / direction** — a variable may not be used as two incompatible classes; an object property
   used backwards is reported with the fix.
4. **Literal datatypes** — an `xsd:string` property compared with a number (or an integer property with a quoted
   string) is flagged.
5. **Grounding** — a string constant compared with a string property (in a triple or in `FILTER(?x = "…")`) must
   occur in the graph; otherwise the closest existing values are suggested.

In [4]:
def iter_nodes(node):
    if isinstance(node, dict):
        yield node
        for v in node.values():
            yield from iter_nodes(v)
    elif isinstance(node, (list, tuple, set)):
        for v in node:
            yield from iter_nodes(v)

VALUES = {}
def values_of(db, prop):
    k = (db, prop)
    if k not in VALUES:
        VALUES[k] = sorted({str(o) for o in GRAPHS[db].objects(None, URIRef(prop)) if isinstance(o, Literal)})
    return VALUES[k]

def check(db, q):
    S = SCHEMA[db]; props = S["Properties"]; classes = set(S["Classes"])
    full = prefixes(db) + q
    try:
        parsed = prepareQuery(full)
    except Exception as e:
        try:
            parseUpdate(full); write = True
        except Exception:
            write = re.match(r"\s*(INSERT|DELETE|DROP|CLEAR|LOAD|CREATE|WITH|ADD|MOVE|COPY)\b", q, re.I) is not None
        if write:
            return ["READ-ONLY GATE: write operations are not permitted; write a SELECT query."]
        return [f"SYNTAX: the query does not parse ({str(e)[:160]})"]
    if parsed.algebra.name not in ("SelectQuery", "AskQuery"):
        return [f"READ-ONLY GATE: {parsed.algebra.name} is not permitted; write a SELECT query."]
    viol, var_cls, var_prop = [], defaultdict(set), {}
    triples = [t for n in iter_nodes(parsed.algebra) if getattr(n, "name", None) == "BGP" for t in n["triples"]]
    RDF_TYPE = URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")
    for s, p, o in triples:
        if p == RDF_TYPE:
            if isinstance(o, URIRef) and str(o) not in classes:
                close = difflib.get_close_matches(str(o), list(classes), n=2)
                viol.append(f"VOCABULARY: class {qname(db, str(o))} does not exist" +
                            (f" (did you mean {', '.join(qname(db, c) for c in close)}?)" if close else "."))
            elif isinstance(s, Variable):
                var_cls[s].add(str(o))
            continue
        if isinstance(p, Variable) or str(p) not in props:
            if not isinstance(p, Variable):
                close = difflib.get_close_matches(str(p), list(props), n=2)
                viol.append(f"VOCABULARY: property {qname(db, str(p))} does not exist" +
                            (f" (did you mean {', '.join(qname(db, c) for c in close)}?)" if close else "."))
            continue
        v = props[str(p)]
        if isinstance(s, Variable):
            var_cls[s].update(v["domain"])
        if is_obj(v["range"]):
            if isinstance(o, Variable):
                var_cls[o].update(v["range"])
            elif isinstance(o, Literal):
                viol.append(f"RANGE: {qname(db, str(p))} links to a {qname(db, v['range'][0])} resource, not to the literal {o.n3()}.")
        else:
            rng = v["range"][0]
            if isinstance(o, Variable):
                var_prop[o] = (str(p), rng)
            elif isinstance(o, Literal):
                viol += literal_issues(db, str(p), rng, o)
    # FILTER(?x = "const") / FILTER(?x = 5) on a datatype property
    for n in iter_nodes(parsed.algebra):
        if getattr(n, "name", None) == "RelationalExpression" and n.get("op") in ("=", "!="):
            a, b = n.get("expr"), n.get("other")
            for var, lit in ((a, b), (b, a)):
                if isinstance(var, Variable) and isinstance(lit, Literal) and var in var_prop:
                    viol += literal_issues(db, var_prop[var][0], var_prop[var][1], lit, in_filter=n.get("op") == "=")
    # domain / range: a variable used as two different classes
    for var, cls in var_cls.items():
        if len(cls) > 1:
            uses = [f"{qname(db, str(p))} ({'subject' if s == var else 'object'})" for s, p, o in triples
                    if var in (s, o) and str(p) in props]
            fixes = [f"{qname(db, str(p))} goes from {qname(db, props[str(p)]['domain'][0])} to {qname(db, props[str(p)]['range'][0])}"
                     for s, p, o in triples if var in (s, o) and str(p) in props and is_obj(props[str(p)]["range"])]
            viol.append(f"DOMAIN/RANGE: {var.n3()} is used as {' and as '.join(sorted(qname(db, c) for c in cls))} "
                        f"in {', '.join(uses)}" + (f". Check the direction: {'; '.join(fixes)}." if fixes else "."))
    return viol

def literal_issues(db, prop, rng, lit, in_filter=True):
    out = []
    dt = lit.datatype
    is_num = dt in (XSD.integer, XSD.decimal, XSD.double, XSD.float) or isinstance(lit.toPython(), (int, float))
    if rng == "string" and is_num:
        out.append(f"DATATYPE: {qname(db, prop)} holds strings; {lit.n3()} is a number. Quote it, e.g. \"{lit}\".")
    elif rng in ("integer", "decimal", "double", "float") and not is_num:
        out.append(f"DATATYPE: {qname(db, prop)} holds numbers ({rng}); {lit.n3()} is a string. Write it without quotes.")
    elif rng == "string" and in_filter:
        vals = values_of(db, prop)
        if vals and str(lit) not in vals:
            close = difflib.get_close_matches(str(lit), vals, n=3, cutoff=0.5) or \
                    [v for v in vals if str(lit).lower() in v.lower()][:3]
            out.append(f"GROUNDING: no resource has {qname(db, prop)} \"{lit}\"." +
                       (f" Closest existing values: {close}." if close else " Check the value or the property."))
    return out

gold_viol = Counter(v.split(":")[0] for q in dev for v in check(q["db_id"], q["sparql"]))
print(f"Checker on the {len(dev)} gold queries: {sum(1 for q in dev if not check(q['db_id'], q['sparql']))} pass;"
      f" violations raised on gold: {dict(gold_viol) or 'none'}")

Checker on the 563 gold queries: 557 pass; violations raised on gold: {'VOCABULARY': 6}


## Round 0: the first query (shared by all three systems)

In [5]:
SYS = ("You translate questions into a single SPARQL SELECT query over an RDF graph with the ontology below. Every "
       "resource has an rdf:type (write `?x a :class`). Return only the query in one ```sparql code block, without "
       "PREFIX lines (they are added for you).\n\nOntology:\n{schema}")

def first_messages(q):
    return [{"role": "system", "content": SYS.format(schema=schema_text(q["db_id"]))},
            {"role": "user", "content": q["question"]}]

def extract_query(text):
    m = re.findall(r"```(?:sparql)?\s*(.*?)```", text, re.S)
    qtxt = (m[-1] if m else text).strip()
    return re.sub(r"(?im)^\s*PREFIX\s+\S*\s*<[^>]*>\s*", "", qtxt).strip()

r0 = llm.run("round0", {q["qid"]: LLM.body(first_messages(q), max_tokens=800) for q in dev})
STATE = {}
for q in dev:
    first = extract_query(r0[q["qid"]]["text"])
    STATE[q["qid"]] = {sysname: {"query": first, "msgs": first_messages(q) + [{"role": "assistant", "content": r0[q['qid']]['text']}],
                                 "repairs": 0, "final": None, "log": []} for sysname in ("one-shot", "retry", "checker")}
print(f"Round 0: {len(r0)} queries")

Round 0: 563 queries


## Rounds 1–3: retry on errors or empty results (b) and ontology-checked repair (c), one Batch job per round

In [6]:
MAX_REPAIRS = 3
def feedback(sysname, q, st):
    """None if the system accepts the current query, else the message to send back."""
    status, out = execute(q["db_id"], st["query"])
    st["exec"] = (status, out)
    if sysname == "one-shot":
        return None
    if sysname == "retry":
        if status != "ok":
            return f"The query failed: {out}. Fix it and return the corrected query."
        return None if out else "The query returned no rows. Check it and return a corrected query."
    viol = check(q["db_id"], st["query"])
    if not viol and status != "ok":
        viol = [f"EXECUTION: {out}"]
    st["log"].append([v.split(":")[0] for v in viol])
    return None if not viol else ("The query violates the ontology:\n- " + "\n- ".join(viol) +
                                  "\nFix these problems and return the corrected query.")

for rnd in range(MAX_REPAIRS + 1):
    reqs = {}
    for q in dev:
        for sysname, st in STATE[q["qid"]].items():
            if st["final"] is not None:
                continue
            fb = feedback(sysname, q, st)
            if fb is None:
                st["final"] = "query"
            elif st["repairs"] == MAX_REPAIRS:
                st["final"] = "unknown" if sysname == "checker" else "query"
            else:
                st["msgs"].append({"role": "user", "content": fb})
                reqs[f"{sysname}|{q['qid']}"] = LLM.body(st["msgs"], max_tokens=800)
    if not reqs:
        break
    res = llm.run(f"repair{rnd + 1}", reqs)
    for k, r in res.items():
        sysname, qid = k.split("|")
        st = STATE[qid][sysname]
        st["msgs"].append({"role": "assistant", "content": r["text"]})
        st["query"] = extract_query(r["text"]); st["repairs"] += 1
    print(f"Round {rnd + 1}: {len(reqs)} repair requests "
          f"({sum(k.startswith('retry') for k in reqs)} retry, {sum(k.startswith('checker') for k in reqs)} checker)")

Round 1: 852 repair requests (377 retry, 475 checker)


Round 2: 420 repair requests (354 retry, 66 checker)


Round 3: 342 repair requests (338 retry, 4 checker)


## Results

In [7]:
def outcome(qid, sysname):
    st = STATE[qid][sysname]
    if st["final"] == "unknown":
        return "unknown"
    status, out = st.get("exec") or execute(next(q for q in dev if q["qid"] == qid)["db_id"], st["query"])
    return "correct" if status == "ok" and same(out, GOLD[qid][1]) else "wrong"

OUT = {(q["qid"], s): outcome(q["qid"], s) for q in dev for s in ("one-shot", "retry", "checker")}
SYSTEMS = ["one-shot", "retry", "checker"]
print(f"{'system':<10}{'correct [95% CI]':>22}{'wrong':>9}{'unknown':>10}{'avg repairs':>13}")
RES = {}
for s in SYSTEMS:
    c = Counter(OUT[(q["qid"], s)] for q in dev); n = len(dev)
    lo, hi = wilson(c["correct"], n)
    RES[s] = (c["correct"], c["wrong"], c["unknown"], n, lo, hi)
    print(f"{s:<10}{f'{c['correct'] / n:.1%} [{lo:.0%}-{hi:.0%}]':>22}"
          f"{c['wrong'] / n:>9.1%}{c['unknown'] / n:>10.1%}{np.mean([STATE[q['qid']][s]['repairs'] for q in dev]):>13.2f}")
acc = lambda s: [float(OUT[(q["qid"], s)] == "correct") for q in dev]
DIFF = {(a, b): paired_boot(acc(a), acc(b)) for a, b in [("one-shot", "retry"), ("retry", "checker"), ("one-shot", "checker")]}
for (a, b), (dm, lo, hi) in DIFF.items():
    print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]")
answered = [q for q in dev if OUT[(q["qid"], "checker")] != "unknown"]
print(f"\nChecker, precision on answered questions: {np.mean([OUT[(q['qid'], 'checker')] == 'correct' for q in answered]):.1%} "
      f"(retry baseline answers everything: {RES['retry'][0] / len(dev):.1%})")
kinds = Counter(k for q in dev for lst in STATE[q["qid"]]["checker"]["log"] for k in lst)
print("Violations found by the checker (all rounds):", dict(kinds.most_common()))
passed_wrong = sum(1 for q in dev if OUT[(q["qid"], "checker")] == "wrong")
print(f"Queries that passed every check but returned wrong rows: {passed_wrong} ({passed_wrong / len(dev):.1%})")

system          correct [95% CI]    wrong   unknown  avg repairs
one-shot         16.7% [14%-20%]    83.3%      0.0%         0.00
retry            22.4% [19%-26%]    77.6%      0.0%         1.90
checker          83.8% [81%-87%]    16.2%      0.0%         0.97
  retry - one-shot: +5.7% [+3.9%, +7.6%]
  checker - retry: +61.5% [+57.0%, +65.7%]
  checker - one-shot: +67.1% [+62.9%, +71.4%]

Checker, precision on answered questions: 83.8% (retry baseline answers everything: 22.4%)
Violations found by the checker (all rounds): {'VOCABULARY': 1360, 'GROUNDING': 7, 'EXECUTION': 3, 'DOMAIN/RANGE': 2, 'SYNTAX': 2}
Queries that passed every check but returned wrong rows: 91 (16.2%)


### A repair the checker made possible

In [8]:
ex = next((q for q in dev if OUT[(q["qid"], "checker")] == "correct" and OUT[(q["qid"], "retry")] != "correct"
           and STATE[q["qid"]]["checker"]["repairs"] > 0), None)
if ex:
    st = STATE[ex["qid"]]["checker"]
    print(f"[{ex['db_id']}] {ex['question']}\n\nFirst query:\n{extract_query(st['msgs'][2]['content'])}\n")
    for m in st["msgs"][3:]:
        if m["role"] == "user":
            print("CHECKER ->", m["content"][:400], "\n")
        else:
            print("REPAIRED ->", extract_query(m["content"])[:400], "\n")

[concert_singer] What is the average, minimum, and maximum age of all singers from France?

First query:
SELECT (AVG(?age) AS ?averageAge) (MIN(?age) AS ?minimumAge) (MAX(?age) AS ?maximumAge)
WHERE {
  ?singer a :singer ;
          :country "France" ;
          :age ?age .
}

CHECKER -> The query violates the ontology:
- VOCABULARY: property :country does not exist (did you mean singer:country, concert:year?)
- VOCABULARY: property :age does not exist (did you mean singer:age, singer:name?)
Fix these problems and return the corrected query. 

REPAIRED -> SELECT (AVG(?age) AS ?averageAge) (MIN(?age) AS ?minimumAge) (MAX(?age) AS ?maximumAge)
WHERE {
  ?singer a :singer ;
          singer:country "France" ;
          singer:age ?age .
} 



## Cost

In [9]:
LEDGER = llm.print_ledger()

stage                           calls     input tok   output tok      USD  via Batch
repair1                           852       515,783       57,084     0.04       100%
repair2                           420       288,081       30,302     0.02       100%
repair3                           342       260,433       25,243     0.02       100%
round0                            563       265,114       34,413     0.02       100%
rule_round0                       563       331,707       34,830     0.03       100%
(embeddings)                        0             0            0     0.00         0%
TOTAL                           2,740     1,661,118      181,872     0.13


In [10]:
print("=" * 92)
print(f"RESULTS SUMMARY — Notebook 17: text-to-SPARQL on Spider4SSC, {len(dev)} questions / {len(DBS)} graphs, gpt-6-luna")
print("=" * 92)
for s in SYSTEMS:
    c, w, u, n, lo, hi = RES[s]
    print(f"{s:<10} correct {c / n:.1%} [{lo:.0%}-{hi:.0%}] | confidently wrong {w / n:.1%} | unknown {u / n:.1%}")
for (a, b), (dm, lo, hi) in DIFF.items():
    print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]" + ("  resolvable" if lo > 0 or hi < 0 else "  not resolvable"))
print(f"Queries that passed every ontology check but were still wrong: {passed_wrong}")
print("\nLecture message (Slides 59-61): the checker's value is measured against a fair baseline that already retries"
      "\non errors and empty results; what it adds is the rows above — fewer confidently wrong answers (some turn into 'unknown') — and"
      "\nwhat it cannot catch is a query that is valid against the ontology but answers a different question.")

RESULTS SUMMARY — Notebook 17: text-to-SPARQL on Spider4SSC, 563 questions / 19 graphs, gpt-6-luna
one-shot   correct 16.7% [14%-20%] | confidently wrong 83.3% | unknown 0.0%
retry      correct 22.4% [19%-26%] | confidently wrong 77.6% | unknown 0.0%
checker    correct 83.8% [81%-87%] | confidently wrong 16.2% | unknown 0.0%
  retry - one-shot: +5.7% [+3.9%, +7.6%]  resolvable
  checker - retry: +61.5% [+57.0%, +65.7%]  resolvable
  checker - one-shot: +67.1% [+62.9%, +71.4%]  resolvable
Queries that passed every ontology check but were still wrong: 91

Lecture message (Slides 59-61): the checker's value is measured against a fair baseline that already retries
on errors and empty results; what it adds is the rows above — fewer confidently wrong answers (some turn into 'unknown') — and
what it cannot catch is a query that is valid against the ontology but answers a different question.


## Added after review: rule-in-prompt baseline

Almost all of the checker's findings (1,360 of 1,374) are vocabulary violations, nearly all of one kind: the model
writes `:age` where Spider4SSC writes `singer:age` (every column lives in its table's namespace). Is the gain due to
*checking*, or only to the information the checker supplies? Two extra systems receive that information up front:

| System | What it gets |
|---|---|
| (d) **Rule prompt** | the round-0 prompt plus explicit naming rules (properties carry their table's prefix; classes use `:`; use only listed terms), one shot, no checker, no repair |
| (e) **Rule prompt + retry** | (d) followed by the error / empty retry of (b), ≤ 3 retries, no checker |

Same 563 questions, same scorer.

*Run note:* the Batch queue was very slow when these cells were added (four requests waited over an hour), so these stages were sent synchronously (`llm.mode = "test"`, standard tier) for whatever was not already cached from Batch; the cache records the tier of every call.

In [11]:
RULES_17 = ("\n\nNaming rules (follow them exactly):\n"
            "1. Every property is written with the prefix of the table (class) it belongs to, exactly as listed in the "
            "ontology, e.g. {ex_p} for the {ex_col} of a {ex_c}. Never write a property with the empty prefix (`:{ex_col}` is wrong).\n"
            "2. Classes are written with the empty prefix, e.g. `?x a {ex_c}`.\n"
            "3. Use only classes and properties that appear in the ontology above; properties in UPPER CASE link to "
            "another resource, the lower-case ..._id properties hold plain integer ids.")

def rule_messages(q):
    db = q["db_id"]
    p, v = next((p, v) for p, v in sorted(SCHEMA[db]["Properties"].items()) if not is_obj(v["range"]))
    ex_p, ex_c = qname(db, p), qname(db, v["domain"][0])
    rules = RULES_17.format(ex_p=ex_p, ex_c=ex_c, ex_col=ex_p.split(":", 1)[1])
    return [{"role": "system", "content": SYS.format(schema=schema_text(db)) + rules},
            {"role": "user", "content": q["question"]}]

if llm.mode == "batch":
    llm.mode = "test"      # synchronous for these small stages (see the run note above)
rr0 = llm.run("rule_round0", {q["qid"]: LLM.body(rule_messages(q), max_tokens=800) for q in dev})
for q in dev:
    first = extract_query(rr0[q["qid"]]["text"])
    for sysname in ("rule-prompt", "rule-retry"):
        STATE[q["qid"]][sysname] = {"query": first, "msgs": rule_messages(q) + [{"role": "assistant", "content": rr0[q["qid"]]["text"]}],
                                    "repairs": 0, "final": None, "log": []}

for rnd in range(MAX_REPAIRS + 1):
    reqs = {}
    for q in dev:
        for sysname in ("rule-prompt", "rule-retry"):
            st = STATE[q["qid"]][sysname]
            if st["final"] is not None:
                continue
            fb = feedback("one-shot" if sysname == "rule-prompt" else "retry", q, st)
            if fb is None or st["repairs"] == MAX_REPAIRS:
                st["final"] = "query"
            else:
                st["msgs"].append({"role": "user", "content": fb})
                reqs[f"{sysname}|{q['qid']}"] = LLM.body(st["msgs"], max_tokens=800)
    if not reqs:
        break
    res = llm.run(f"rule_repair{rnd + 1}", reqs)
    for k, r in res.items():
        sysname, qid = k.split("|")
        st = STATE[qid][sysname]
        st["msgs"].append({"role": "assistant", "content": r["text"]})
        st["query"] = extract_query(r["text"]); st["repairs"] += 1
    print(f"Round {rnd + 1}: {len(reqs)} rule-retry requests")

ALL5 = SYSTEMS + ["rule-prompt", "rule-retry"]
for q in dev:
    for s in ("rule-prompt", "rule-retry"):
        OUT[(q["qid"], s)] = outcome(q["qid"], s)
print(f"{'system':<12}{'correct [95% CI]':>22}{'avg repairs':>13}")
for s in ALL5:
    k = sum(OUT[(q["qid"], s)] == "correct" for q in dev); lo, hi = wilson(k, len(dev))
    print(f"{s:<12}{f'{k / len(dev):.1%} [{lo:.1%}-{hi:.1%}]':>22}{np.mean([STATE[q['qid']][s]['repairs'] for q in dev]):>13.2f}")
DIFF2 = {(a, b): paired_boot(acc(a), acc(b)) for a, b in [("one-shot", "rule-prompt"), ("retry", "rule-retry"),
                                                           ("rule-prompt", "checker"), ("rule-retry", "checker")]}
for (a, b), (dm, lo, hi) in DIFF2.items():
    print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]" + ("  resolvable" if lo > 0 or hi < 0 else "  not resolvable"))
nov = lambda s: sum(1 for q in dev if any(v.startswith("VOCABULARY") for v in check(q["db_id"], STATE[q["qid"]][s]["query"])))
print(f"\nFinal queries with a vocabulary violation: " + ", ".join(f"{s} {nov(s)}" for s in ALL5))
llm.print_ledger(sorted(f[:-6] for f in os.listdir(llm.cache) if f.startswith("rule_") and f.endswith(".jsonl")))

[rule_round0] 4 new requests -> test (559 cached)


[rule_repair1] 32 new requests -> test (0 cached)


Round 1: 32 rule-retry requests


[rule_repair2] 13 new requests -> test (0 cached)


Round 2: 13 rule-retry requests


[rule_repair3] 12 new requests -> test (0 cached)


Round 3: 12 rule-retry requests


system            correct [95% CI]  avg repairs
one-shot       16.7% [13.8%-20.0%]         0.00
retry          22.4% [19.1%-26.0%]         1.90
checker        83.8% [80.6%-86.6%]         0.97
rule-prompt    78.9% [75.3%-82.0%]         0.00
rule-retry     81.9% [78.5%-84.8%]         0.10
  rule-prompt - one-shot: +62.2% [+57.7%, +66.4%]  resolvable
  rule-retry - retry: +59.5% [+55.1%, +63.9%]  resolvable
  checker - rule-prompt: +5.0% [+2.5%, +7.6%]  resolvable
  checker - rule-retry: +2.0% [-0.5%, +4.4%]  not resolvable



Final queries with a vocabulary violation: one-shot 473, retry 440, checker 0, rule-prompt 21, rule-retry 12
stage                           calls     input tok   output tok      USD  via Batch
rule_repair1                       32        21,405        2,644     0.00         0%
rule_repair2                       13         9,593        1,291     0.00         0%
rule_repair3                       12        10,123        1,220     0.00         0%
rule_round0                       563       334,099       35,119     0.03        99%
(embeddings)                        0             0            0     0.00         0%
TOTAL                             620       375,220       40,274     0.03


{'rule_repair1': {'calls': 32,
  'in': 21405,
  'out': 2644,
  'usd': 0.0034625000000000003,
  'batch_share': 0.0},
 'rule_repair2': {'calls': 13,
  'in': 9593,
  'out': 1291,
  'usd': 0.0016048000000000002,
  'batch_share': 0.0},
 'rule_repair3': {'calls': 12,
  'in': 10123,
  'out': 1220,
  'usd': 0.0016223,
  'batch_share': 0.0},
 'rule_round0': {'calls': 563,
  'in': 334099,
  'out': 35119,
  'usd': 0.02567654999999997,
  'batch_share': 0.9928952042628775},
 '(embeddings)': {'calls': 0,
  'in': 0,
  'out': 0,
  'usd': 0.0,
  'batch_share': 0.0}}